## RAW ANIME Dataset

This dataset contains comprehensive details of 24,905 anime entries.

It offers valuable information for analyzing and comprehending the characteristics, ratings, popularity, and viewership of various anime shows. By utilizing this dataset, one can conduct a wide range of analyses, including identifying the highest-rated anime, exploring the most popular genres, examining the distribution of ratings, and gaining insights into viewer preferences and trends. 


Reading the imported dataset

In [0]:
spark.read.table("hive_metastore.anime_raw.raw_dataset").display()

In [0]:
df_raw = spark.read.table("hive_metastore.anime_raw.raw_dataset")
df_raw.count()


Checking schema and columns

In [0]:
df_raw.printSchema()
df_raw.columns

Removing whitespace

In [0]:

from pyspark.sql import functions as sf

df_raw = df_raw.select([sf.trim(col).alias(col) for col in df_raw.columns])
display(df_raw)

Droping duplicates

In [0]:
df_raw = df_raw.dropDuplicates()
df_raw.count()

Drop rows where all specified columns are NULL

In [0]:

df_raw = df_raw.na.drop(how="all")
display(df_raw)


In [0]:
from pyspark.sql import functions as F

# Iterate over all columns in the DataFrame and apply length and trim operations
for col in df_raw.columns:
    df_raw = df_raw.filter(
        F.length(col) != F.length(F.trim(F.col(col)))
    )

# Count the rows that needed trimming
df_count = df_raw.count()
print(f"Rows that needed trimming: {df_count}")

Iterating over all columns in the DataFrame df_raw, replacing invalid characters with underscores in the column names

In [0]:
# Rename columns to remove invalid characters
df_clean = df_raw
for col in df_raw.columns:
    new_col = col.replace(" ", "_").replace(",", "_").replace(";", "_").replace("{", "_") \
                 .replace("}", "_").replace("(", "_").replace(")", "_").replace("\n", "_") \
                 .replace("\t", "_").replace("=", "_")
    df_clean = df_clean.withColumnRenamed(col, new_col)


Expected output to Silver: 
- Remove whitespace
- Remove duplicates
- Drop rows where all specified columns are NULL
- Replace invalid characters with underscores in the column names

In [0]:
display(df_clean)

Saving to Silver

In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable("hive_metastore.anime_silver.silver_dataset")